In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../../data/processed")

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ]
)

In [3]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000
)

In [4]:
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logistic_model)
    ]
)

In [5]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

setup = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

In [6]:
print(cv_results)

{'fit_time': array([1.9225359 , 1.68841124, 1.37610269, 2.02829623, 2.44657326]), 'score_time': array([0.08750582, 0.05513263, 0.09196067, 0.11200595, 0.05427599]), 'test_roc_auc': array([0.85053561, 0.8361495 , 0.84171163, 0.84204727, 0.84064103]), 'test_precision': array([0.62908012, 0.64488636, 0.66451613, 0.64556962, 0.63389831]), 'test_recall': array([0.24508671, 0.26242775, 0.23815029, 0.23556582, 0.21593533]), 'test_f1': array([0.35274542, 0.37304848, 0.3506383 , 0.34517766, 0.32213609])}


In [7]:
print("ROC-AUC:", cv_results["test_roc_auc"].mean())
print("Precision:", cv_results["test_precision"].mean())
print("Recall:", cv_results["test_recall"].mean())
print("F1:", cv_results["test_f1"].mean())

ROC-AUC: 0.8422170074520186
Precision: 0.6435901073401789
Recall: 0.2394331789237608
F1: 0.3487491913172568


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

balanced_logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", balanced_logistic_model)
    ]
)

balanced_results = cross_validate(
    balanced_logistic_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", balanced_results["test_roc_auc"].mean())
print("Precision:", balanced_results["test_precision"].mean())
print("Recall:", balanced_results["test_recall"].mean())
print("F1:", balanced_results["test_f1"].mean())

ROC-AUC: 0.8433316790258107
Precision: 0.3107107395510522
Recall: 0.7409309962754809
F1: 0.4377402971236329


In [9]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

train_probabilities = cross_val_predict(
    logistic_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    method="predict_proba"
)[:, 1]

thresholds = [0.5, 0.4, 0.3, 0.2]

for threshold in thresholds:
    predictions = (train_probabilities >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print("Precision:", precision_score(y_train, predictions))
    print("Recall:", recall_score(y_train, predictions))
    print("F1:", f1_score(y_train, predictions))
    print()

Threshold: 0.5
Precision: 0.6434782608695652
Recall: 0.2394268546336954
F1: 0.3489978103419235

Threshold: 0.4
Precision: 0.5792988313856428
Recall: 0.3207765195285417
F1: 0.4129109028707422

Threshold: 0.3
Precision: 0.4967828418230563
Recall: 0.428241275710654
F1: 0.45997269455132184

Threshold: 0.2
Precision: 0.4000973078170613
Recall: 0.5701409752715507
F1: 0.47021824073191654



In [10]:
balanced_probabilities = cross_val_predict(
    balanced_logistic_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    method="predict_proba"
)[:, 1]

thresholds = [0.5, 0.4, 0.3, 0.2]

for threshold in thresholds:
    predictions = (balanced_probabilities >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print("Precision:", precision_score(y_train, predictions))
    print("Recall:", recall_score(y_train, predictions))
    print("F1:", f1_score(y_train, predictions))
    print()

Threshold: 0.5
Precision: 0.31059872117806625
Recall: 0.7409290501502196
F1: 0.43770905863881493

Threshold: 0.4
Precision: 0.26460280373831774
Recall: 0.8375317772128495
F1: 0.4021528047494868

Threshold: 0.3
Precision: 0.21876396960214572
Recall: 0.9047839149526231
F1: 0.35233766818161366

Threshold: 0.2
Precision: 0.17946282062354263
Recall: 0.9604807025652877
F1: 0.302419501546298



In [11]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

balanced_probabilities = cross_val_predict(
    balanced_logistic_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    method="predict_proba"
)[:, 1]

thresholds = [0.5, 0.6, 0.7, 0.8]

for threshold in thresholds:
    predictions = (balanced_probabilities >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print("Precision:", precision_score(y_train, predictions))
    print("Recall:", recall_score(y_train, predictions))
    print("F1:", f1_score(y_train, predictions))
    print()

Threshold: 0.5
Precision: 0.31059872117806625
Recall: 0.7409290501502196
F1: 0.43770905863881493

Threshold: 0.6
Precision: 0.36255454184847286
Recall: 0.6336954009706494
F1: 0.4612279226240538

Threshold: 0.7
Precision: 0.43082178781552477
Recall: 0.5246128957707419
F1: 0.47311379741558984

Threshold: 0.8
Precision: 0.5144927536231884
Recall: 0.3938063323318697
F1: 0.44613169262992536



## Logistic Regression

Logistic Regression was used as the first baseline classification model for ICU mortality prediction.

- Numerical features were processed with median imputation followed by `StandardScaler`.
- Categorical features were transformed using `OneHotEncoder`.
- Preprocessing and Logistic Regression were combined in a single pipeline.
- Model performance was evaluated using 5-fold `StratifiedGroupKFold`, keeping ICU stays from the same patient in the same fold.
- The baseline model achieved approximately:
  - ROC-AUC: **0.84**
  - Precision: **0.64**
  - Recall: **0.24**
  - F1: **0.35**
- Because mortality represented the minority class, `class_weight="balanced"` was also tested.
- The balanced model increased recall to approximately **0.74**, but reduced precision to approximately **0.31**.
- Threshold tuning was then applied to both models.
- For the standard Logistic Regression, a threshold of **0.20** provided a more suitable trade-off for this clinical task:
  - Precision: **0.40**
  - Recall: **0.57**
  - F1: **0.47**
- For the balanced Logistic Regression, a threshold of **0.70** produced:
  - Precision: **0.43**
  - Recall: **0.52**
  - F1: **0.47**
- Since identifying high-risk patients is especially important in this project, the standard Logistic Regression with a **0.20 threshold** was retained as the stronger Logistic Regression candidate due to its higher recall.